## Week 2 Day 2

Our first Agentic Framework project!!

Prepare yourself for something ridiculously easy.

We're going to build a simple Agent system for generating cold sales outreach emails:
1. Agent workflow
2. Use of tools to call functions
3. Agent collaboration via Tools and Handoffs

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/stop.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">IMPORTANT PLEASE READ - Email setup</h2>
            <span style="color:#ff7800;">We use Google SMTP for sending emails (no SendGrid rate limits).<br/>
            The next cell contains instructions for SMTP setup.<br/>
            Make sure your .env has SMTP_SERVER_URL, SMTP_SERVER_PORT, SMTP_SERVER_USERNAME, and SMTP_SERVER_PASSWORD
            </span>
        </td>
    </tr>
</table>

## Setting up Google SMTP

Your .env file already has these SMTP variables:
- SMTP_SERVER_URL (e.g. smtp.gmail.com)
- SMTP_SERVER_PORT (e.g. 587)
- SMTP_SERVER_USERNAME (your email)
- SMTP_SERVER_PASSWORD (your app password)

For Gmail, generate an App Password at:
https://myaccount.google.com/apppasswords

You need:
1. 2-Factor Authentication enabled on your Google account
2. An App Password generated and saved in your .env


In [27]:
from dotenv import load_dotenv
from openai import AsyncOpenAI
from agents import Agent, Runner, trace, function_tool, OpenAIChatCompletionsModel, set_default_openai_client
from openai.types.responses import ResponseTextDeltaEvent
from typing import Dict
import os
import smtplib
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart
import asyncio
import json
from datetime import datetime

In [28]:
load_dotenv(override=True)

True

In [29]:
# Configure DeepSeek as the model provider

DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY") or os.getenv("OPENAI_API_KEY")
DEEPSEEK_BASE_URL = os.getenv("DEEPSEEK_OPEN_API_ENDPOINT") or "https://api.deepseek.com/v1"

def _normalize_openai_base_url(url: str) -> str:
    url = url.rstrip("/")
    if not url.endswith("/v1"):
        url = f"{url}/v1"
    return url

DEEPSEEK_BASE_URL = _normalize_openai_base_url(DEEPSEEK_BASE_URL)

if not DEEPSEEK_API_KEY:
    raise RuntimeError("Missing DEEPSEEK_API_KEY or OPENAI_API_KEY in .env file")

print("DeepSeek base URL:", DEEPSEEK_BASE_URL)
print("DeepSeek model: deepseek-v4-flash")

deepseek_client = AsyncOpenAI(api_key=DEEPSEEK_API_KEY, base_url=DEEPSEEK_BASE_URL)
deepseek_model = OpenAIChatCompletionsModel(model="deepseek-v4-flash", openai_client=deepseek_client)
set_default_openai_client(deepseek_client)


DeepSeek base URL: https://api.deepseek.com/v1
DeepSeek model: deepseek-v4-flash


In [30]:
# ──────────────────────────────────────────────────────────
# OPTION B: Arize Phoenix – Visual trace dashboard
# ──────────────────────────────────────────────────────────
# DeepSeek doesn't support OpenAI's cloud-based traces at
# platform.openai.com/traces.  Instead we use Arize Phoenix,
# a free self-hosted visual trace dashboard.
#
# After running this cell, visit http://localhost:6006
# to see a full trace explorer for all Agent runs.
#
# Prerequisites: Ensure the Phoenix server is already running.
# Run this in a terminal:
#   uvx arize-phoenix serve
#
# Install:  uv pip install arize-phoenix-otel openinference-instrumentation-openai-agents

from opentelemetry import trace as otel_trace
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import BatchSpanProcessor
from opentelemetry.exporter.otlp.proto.http.trace_exporter import OTLPSpanExporter
from openinference.instrumentation.openai_agents import OpenAIAgentsInstrumentor

# Configure OpenTelemetry to export traces to the local Phoenix server
tracer_provider = TracerProvider()
otlp_exporter = OTLPSpanExporter(endpoint="http://localhost:6006/v1/traces")
tracer_provider.add_span_processor(BatchSpanProcessor(otlp_exporter))
otel_trace.set_tracer_provider(tracer_provider)

# Instrument the OpenAI Agents SDK
OpenAIAgentsInstrumentor().instrument(tracer_provider=tracer_provider)

print("✅ Arize Phoenix trace dashboard active at http://localhost:6006")
print("All trace() calls will now be visible in the Phoenix UI.")

Overriding of current TracerProvider is not allowed
Attempting to instrument while already instrumented


✅ Arize Phoenix trace dashboard active at http://localhost:6006
All trace() calls will now be visible in the Phoenix UI.


In [31]:
# Let's just check emails are working for you

def send_test_email():
    smtp_host = os.environ.get('SMTP_SERVER_URL', 'smtp.gmail.com')
    smtp_port = int(os.environ.get('SMTP_SERVER_PORT', '587'))
    smtp_user = os.environ.get('SMTP_SERVER_USERNAME')
    smtp_pass = os.environ.get('SMTP_SERVER_PASSWORD')
    from_addr = smtp_user or os.environ.get('SMTP_SERVER_EMAIL')
    to_addr = from_addr

    msg = MIMEText("This is an important test email")
    msg['Subject'] = "Test email"
    msg['From'] = from_addr
    msg['To'] = to_addr

    with smtplib.SMTP(smtp_host, smtp_port) as server:
        server.starttls()
        server.login(smtp_user, smtp_pass)
        server.send_message(msg)
    print("Email sent successfully")

send_test_email()

Email sent successfully


### Did you receive the test email

If you receive the email, you're good to go!

#### Certificate error

If you get an error SSL: CERTIFICATE_VERIFY_FAILED then students Chris S and Oleksandr K have suggestions:  
First run this: `!uv pip install --upgrade certifi`  
Next, run this:
```python
import certifi
import os
os.environ['SSL_CERT_FILE'] = certifi.where()
```

#### Other errors

If there are other problems, you'll need to check your API key and your verified sender email address in the SendGrid dashboard



(Or - you could always replace the email sending code below with a Pushover call, or something to simply write to a flat file)

## Step 1: Agent workflow

In [32]:
instructions1 = "You are a sales agent working for ComplAI, \
a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI. \
You write professional, serious cold emails."

instructions2 = "You are a humorous, engaging sales agent working for ComplAI, \
a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI. \
You write witty, engaging cold emails that are likely to get a response."

instructions3 = "You are a busy sales agent working for ComplAI, \
a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI. \
You write concise, to the point cold emails."

In [33]:
sales_agent1 = Agent(
        name="Professional Sales Agent",
        instructions=instructions1,
        model=deepseek_model
)

sales_agent2 = Agent(
        name="Engaging Sales Agent",
        instructions=instructions2,
        model=deepseek_model
)

sales_agent3 = Agent(
        name="Busy Sales Agent",
        instructions=instructions3,
        model=deepseek_model
)

In [34]:

result = Runner.run_streamed(sales_agent1, input="Write a cold sales email")
async for event in result.stream_events():
    if event.type == "raw_response_event" and isinstance(event.data, ResponseTextDeltaEvent):
        print(event.data.delta, end="", flush=True)

**Subject:** Simplify SOC2 compliance with AI-powered automation  

Dear [First Name],  

Managing SOC2 compliance and audit readiness can be a complex, time-consuming process—especially when your team is already stretched thin. Manual evidence collection, policy tracking, and control monitoring often distract from core business priorities and create unnecessary stress as audits approach.  

At ComplAI, we’ve built a SaaS solution that leverages artificial intelligence to streamline and automate the entire SOC2 compliance lifecycle. From continuous control monitoring to intelligent evidence collection and audit-ready reporting, our platform reduces manual effort by up to 70% and gives your team confidence that you’re always audit-ready.  

Unlike traditional compliance tools, ComplAI’s AI engine proactively identifies gaps, suggests remediation steps, and learns from your environment—so you spend less time on paperwork and more time on growth.  

I’d be happy to schedule a brief call t

In [35]:
message = "Write a cold sales email"

with trace("Parallel cold emails"):
    results = await asyncio.gather(
        Runner.run(sales_agent1, message),
        Runner.run(sales_agent2, message),
        Runner.run(sales_agent3, message),
    )

outputs = [result.final_output for result in results]

for output in outputs:
    print(output + "\n\n")


**Subject:** Simplify SOC2 Compliance with AI-Powered Automation  

Dear [Prospect's Name],  

Managing SOC2 compliance can be a time-intensive process, especially when audits loom. At ComplAI, we help companies like yours streamline compliance and audit preparation with an AI-driven SaaS platform that automates evidence collection, policy monitoring, and readiness assessments.  

Our solution reduces manual effort by up to 80%, enabling your team to focus on core business priorities while staying audit-ready year-round. With real-time gap analysis and intelligent documentation, ComplAI ensures you’re never caught off guard by an upcoming audit.  

I’d love to schedule a brief call to explore how ComplAI can simplify your compliance journey. Would you be available for 15 minutes next week?  

Best regards,  
[Your Name]  
Sales Agent, ComplAI  
[Email] | [Phone] | [Website]


**Subject:** Audits: the only thing worse than a root canal (and we can fix that)

Hi [First Name],

Let’s be h

In [36]:
sales_picker = Agent(
    name="sales_picker",
    instructions="You pick the best cold sales email from the given options. \
Imagine you are a customer and pick the one you are most likely to respond to. \
Do not give an explanation; reply with the selected email only.",
    model=deepseek_model
)

In [37]:
message = "Write a cold sales email"

with trace("Selection from sales people"):
    results = await asyncio.gather(
        Runner.run(sales_agent1, message),
        Runner.run(sales_agent2, message),
        Runner.run(sales_agent3, message),
    )
    outputs = [result.final_output for result in results]

    emails = "Cold sales emails:\n\n" + "\n\nEmail:\n\n".join(outputs)

    best = await Runner.run(sales_picker, emails)

    print(f"Best sales email:\n{best.final_output}")


Best sales email:
Email:

Subject: SOC2 audits: the gift that keeps on giving (until now)

Hi [First Name],

Let me guess: your compliance checklist is currently serving as a paperweight, your auditors have the personality of a spreadsheet, and somewhere in your Slack a developer just asked, “Wait, do we _really_ need to encrypt the snack fridge logs?”

We get it. SOC2 compliance is like that one relative who shows up unannounced, eats all your snacks, and then critiques your life choices. But you can’t just ghost them—your customers (and their contracts) expect you to be audit-ready.

That’s where ComplAI comes in. We’re the AI-powered sidekick that makes SOC2 prep feel less like a root canal and more like… okay, still compliance, but _fast_ and _fun_. We automate evidence collection, map controls to your actual infrastructure, and whisper sweet nothings to your auditor’s checklist. 

No more manually exporting logs at 2 a.m. No more “accidentally” losing the encryption policy PDF. Ju

### Traces have been saved locally

Check the `./traces/` directory for JSON files with full agent execution traces.

Alternatively you can view them with a visual dashboard – see the finale section at the bottom of this notebook.

## Part 2: use of tools

Now we will add a tool to the mix.

Remember all that json boilerplate and the `handle_tool_calls()` function with the if logic..

In [39]:
sales_agent1 = Agent(
        name="Professional Sales Agent",
        instructions=instructions1,
        model=deepseek_model,
)

sales_agent2 = Agent(
        name="Engaging Sales Agent",
        instructions=instructions2,
        model=deepseek_model,
)

sales_agent3 = Agent(
        name="Busy Sales Agent",
        instructions=instructions3,
        model=deepseek_model,
)

In [40]:
sales_agent1

Agent(name='Professional Sales Agent', handoff_description=None, tools=[], mcp_servers=[], mcp_config={}, instructions='You are a sales agent working for ComplAI, a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI. You write professional, serious cold emails.', prompt=None, handoffs=[], model=<agents.models.openai_chatcompletions.OpenAIChatCompletionsModel object at 0x1131661b0>, model_settings=ModelSettings(temperature=None, top_p=None, frequency_penalty=None, presence_penalty=None, tool_choice=None, parallel_tool_calls=None, truncation=None, max_tokens=None, reasoning=None, verbosity=None, metadata=None, store=None, include_usage=None, response_include=None, top_logprobs=None, extra_query=None, extra_body=None, extra_headers=None, extra_args=None), input_guardrails=[], output_guardrails=[], output_type=None, hooks=None, tool_use_behavior='run_llm_again', reset_tool_choice=True)

## Steps 2 and 3: Tools and Agent interactions

Remember all that boilerplate json?

Simply wrap your function with the decorator `@function_tool`

In [42]:
@function_tool
def send_email(body: str):
    """ Send out an email with the given body to all sales prospects """
    smtp_host = os.environ.get('SMTP_SERVER_URL', 'smtp.gmail.com')
    smtp_port = int(os.environ.get('SMTP_SERVER_PORT', '587'))
    smtp_user = os.environ.get('SMTP_SERVER_USERNAME')
    smtp_pass = os.environ.get('SMTP_SERVER_PASSWORD')
    from_addr = smtp_user or os.environ.get('SMTP_SERVER_EMAIL')
    to_addr = from_addr  # Send to self; change as needed

    msg = MIMEText(body)
    msg['Subject'] = "Sales email"
    msg['From'] = from_addr
    msg['To'] = to_addr

    with smtplib.SMTP(smtp_host, smtp_port) as server:
        server.starttls()
        server.login(smtp_user, smtp_pass)
        server.send_message(msg)
    return {"status": "success"}

### This has automatically been converted into a tool, with the boilerplate json created

In [43]:
# Let's look at it
send_email

FunctionTool(name='send_email', description='Send out an email with the given body to all sales prospects', params_json_schema={'properties': {'body': {'title': 'Body', 'type': 'string'}}, 'required': ['body'], 'title': 'send_email_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x11306eb60>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None)

### And you can also convert an Agent into a tool

In [44]:
tool1 = sales_agent1.as_tool(tool_name="sales_agent1", tool_description="Write a cold sales email")
tool1

FunctionTool(name='sales_agent1', description='Write a cold sales email', params_json_schema={'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'sales_agent1_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x11306f060>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None)

### So now we can gather all the tools together:

A tool for each of our 3 email-writing agents

And a tool for our function to send emails

In [45]:
description = "Write a cold sales email"

tool1 = sales_agent1.as_tool(tool_name="sales_agent1", tool_description=description)
tool2 = sales_agent2.as_tool(tool_name="sales_agent2", tool_description=description)
tool3 = sales_agent3.as_tool(tool_name="sales_agent3", tool_description=description)

tools = [tool1, tool2, tool3, send_email]

tools

[FunctionTool(name='sales_agent1', description='Write a cold sales email', params_json_schema={'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'sales_agent1_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x11306f100>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None),
 FunctionTool(name='sales_agent2', description='Write a cold sales email', params_json_schema={'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'sales_agent2_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x11306e7a0>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None),
 FunctionTool(name='sales_agent3', description='Write 

## And now it's time for our Sales Manager - our planning agent

In [46]:
# Improved instructions thanks to student Guillermo F.

instructions = """
You are a Sales Manager at ComplAI. Your goal is to find the single best cold sales email using the sales_agent tools.
 
Follow these steps carefully:
1. Generate Drafts: Use all three sales_agent tools to generate three different email drafts. Do not proceed until all three drafts are ready.
 
2. Evaluate and Select: Review the drafts and choose the single best email using your judgment of which one is most effective.
 
3. Use the send_email tool to send the best email (and only the best email) to the user.
 
Crucial Rules:
- You must use the sales agent tools to generate the drafts — do not write them yourself.
- You must send ONE email using the send_email tool — never more than one.
"""


sales_manager = Agent(name="Sales Manager", instructions=instructions, tools=tools, model=deepseek_model)

message = "Send a cold sales email addressed to 'Dear CEO'"

with trace("Sales manager"):
    result = await Runner.run(sales_manager, message)

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/stop.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Wait - you didn't get an email??</h2>
            <span style="color:#ff7800;">With much thanks to student Chris S. for describing his issue and fixes. 
            If you don't receive an email after running the prior cell, here are some things to check: <br/>
            First, check your Spam folder! Several students have missed that the emails arrived in Spam!<br/>Second, print(result) and see if you are receiving errors about SSL. 
            If you're receiving SSL errors, then please check out theses <a href="https://chatgpt.com/share/680620ec-3b30-8012-8c26-ca86693d0e3d">networking tips</a> and see the note in the next cell. Also look at the trace in OpenAI, and investigate on the SendGrid website, to hunt for clues. Let me know if I can help!
            </span>
        </td>
    </tr>
</table>

### And one more suggestion to send emails from student Oleksandr on Windows 11:

If you are getting certificate SSL errors, then:  
Run this in a terminal: `uv pip install --upgrade certifi`

Then run this code:
```python
import certifi
import os
os.environ['SSL_CERT_FILE'] = certifi.where()
```

Thank you Oleksandr!

## Check local traces

Open the `./traces/` folder – each agent run produced a JSON file with the full trace.
You can also view them visually with the `openinference-inspector` tool (see the final section).

### Handoffs represent a way an agent can delegate to an agent, passing control to it

Handoffs and Agents-as-tools are similar:

In both cases, an Agent can collaborate with another Agent

With tools, control passes back

With handoffs, control passes across



In [47]:

subject_instructions = "You can write a subject for a cold sales email. \
You are given a message and you need to write a subject for an email that is likely to get a response."

html_instructions = "You can convert a text email body to an HTML email body. \
You are given a text email body which might have some markdown \
and you need to convert it to an HTML email body with simple, clear, compelling layout and design."

subject_writer = Agent(name="Email subject writer", instructions=subject_instructions, model=deepseek_model)
subject_tool = subject_writer.as_tool(tool_name="subject_writer", tool_description="Write a subject for a cold sales email")

html_converter = Agent(name="HTML email body converter", instructions=html_instructions, model=deepseek_model)
html_tool = html_converter.as_tool(tool_name="html_converter",tool_description="Convert a text email body to an HTML email body")


In [49]:
@function_tool
def send_html_email(subject: str, html_body: str) -> Dict[str, str]:
    """ Send out an email with the given subject and HTML body to all sales prospects """
    smtp_host = os.environ.get('SMTP_SERVER_URL', 'smtp.gmail.com')
    smtp_port = int(os.environ.get('SMTP_SERVER_PORT', '587'))
    smtp_user = os.environ.get('SMTP_SERVER_USERNAME')
    smtp_pass = os.environ.get('SMTP_SERVER_PASSWORD')
    from_addr = smtp_user or os.environ.get('SMTP_SERVER_EMAIL')
    to_addr = from_addr  # Send to self; change as needed

    msg = MIMEMultipart('alternative')
    msg['Subject'] = subject
    msg['From'] = from_addr
    msg['To'] = to_addr
    part = MIMEText(html_body, 'html')
    msg.attach(part)

    with smtplib.SMTP(smtp_host, smtp_port) as server:
        server.starttls()
        server.login(smtp_user, smtp_pass)
        server.send_message(msg)
    return {"status": "success"}

In [50]:
tools = [subject_tool, html_tool, send_html_email]

In [51]:
tools

[FunctionTool(name='subject_writer', description='Write a subject for a cold sales email', params_json_schema={'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'subject_writer_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x11306fe20>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None),
 FunctionTool(name='html_converter', description='Convert a text email body to an HTML email body', params_json_schema={'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'html_converter_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x11306fb00>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None),
 Function

In [52]:
instructions ="You are an email formatter and sender. You receive the body of an email to be sent. \
You need to update the sale agent name as John Doe in the email body. \
You first use the subject_writer tool to write a subject for the email, then use the html_converter tool to convert the body to HTML. \
Finally, you use the send_html_email tool to send the email with the subject and HTML body."


emailer_agent = Agent(
    name="Email Manager",
    instructions=instructions,
    tools=tools,
    model=deepseek_model,
    handoff_description="Convert an email to HTML and send it")


### Now we have 3 tools and 1 handoff

In [53]:
tools = [tool1, tool2, tool3]
handoffs = [emailer_agent]
print(tools)
print(handoffs)

[FunctionTool(name='sales_agent1', description='Write a cold sales email', params_json_schema={'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'sales_agent1_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x11306f100>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None), FunctionTool(name='sales_agent2', description='Write a cold sales email', params_json_schema={'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'sales_agent2_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x11306e7a0>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None), FunctionTool(name='sales_agent3', description='Write a 

In [54]:
# Improved instructions thanks to student Guillermo F.

sales_manager_instructions = """
You are a Sales Manager at ComplAI. Your goal is to find the single best cold sales email using the sales_agent tools.
 
Follow these steps carefully:
1. Generate Drafts: Use all three sales_agent tools to generate three different email drafts. Do not proceed until all three drafts are ready.
 
2. Evaluate and Select: Review the drafts and choose the single best email using your judgment of which one is most effective.
You can use the tools multiple times if you're not satisfied with the results from the first try.
 
3. Handoff for Sending: Pass ONLY the winning email draft to the 'Email Manager' agent. The Email Manager will take care of formatting and sending.
 
Crucial Rules:
- You must use the sales agent tools to generate the drafts — do not write them yourself.
- You must hand off exactly ONE email to the Email Manager — never more than one.
"""


sales_manager = Agent(
    name="Sales Manager",
    instructions=sales_manager_instructions,
    tools=tools,
    handoffs=handoffs,
    model=deepseek_model)

message = "Send out a cold sales email addressed to Dear CEO from Alice"

with trace("Automated SDR"):
    result = await Runner.run(sales_manager, message)
result

RunResult(input='Send out a cold sales email addressed to Dear CEO from Alice', new_items=[ReasoningItem(agent=Agent(name='Sales Manager', handoff_description=None, tools=[FunctionTool(name='sales_agent1', description='Write a cold sales email', params_json_schema={'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'sales_agent1_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x11306f100>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None), FunctionTool(name='sales_agent2', description='Write a cold sales email', params_json_schema={'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'sales_agent2_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool a

## Check local traces

Open the `./traces/` folder – the `Automated SDR` trace was saved as a JSON file.


---

## 🎯 Arize Phoenix – Visual trace dashboard

Use the free [Arize Phoenix](https://github.com/Arize-ai/phoenix) project for a visual UI of your agent traces.

### 1. Start the Phoenix server (in a terminal)

```bash
uvx arize-phoenix serve
```

This starts a dashboard at **http://localhost:6006**.

### 2. Run Cell 6 in this notebook

Cell 6 configures OpenTelemetry to export traces to the local Phoenix server.
Already done? Cell 6 will print "✅ Arize Phoenix trace dashboard active..."

### 3. Run your agent cells

Every `with trace("..."):` block and `Runner.run()` call will appear as a trace in Phoenix.

---

### Summary of 3rd-party tracing options

| Option | What it does | Requires |
|--------|-------------|----------|
| **A – Local JSON** (used above) | Saves traces to `./traces/*.json` | Nothing extra |
| **B – Arize Phoenix** | Visual dashboard at `localhost:6006` | `uvx arize-phoenix serve` + run Cell 6 |
| **C – Langfuse** (commercial) | Cloud-based trace viewer | API key from langfuse.com |
| **D – Custom processor** | Send traces to your own backend | Implement `TracingProcessor` interface |

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Exercise</h2>
            <span style="color:#ff7800;">Can you identify the Agentic design patterns that were used here?<br/>
            What is the 1 line that changed this from being an Agentic "workflow" to "agent" under Anthropic's definition?<br/>
            Try adding in more tools and Agents! You could have tools that handle the mail merge to send to a list.<br/><br/>
            HARD CHALLENGE: research how you can have SendGrid call a Callback webhook when a user replies to an email,
            Then have the SDR respond to keep the conversation going! This may require some "vibe coding" 😂
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#00bfff;">Commercial implications</h2>
            <span style="color:#00bfff;">This is immediately applicable to Sales Automation; but more generally this could be applied to  end-to-end automation of any business process through conversations and tools. Think of ways you could apply an Agent solution
            like this in your day job.
            </span>
        </td>
    </tr>
</table>

## Extra note:

Google has released their Agent Development Kit (ADK). It's not yet got the traction of the other frameworks on this course, but it's getting some attention. It's interesting to note that it looks quite similar to OpenAI Agents SDK. To give you a preview, here's a peak at sample code from ADK:

```
root_agent = Agent(
    name="weather_time_agent",
    model="gemini-2.0-flash",
    description="Agent to answer questions about the time and weather in a city.",
    instruction="You are a helpful agent who can answer user questions about the time and weather in a city.",
    tools=[get_weather, get_current_time]
)
```

Well, that looks familiar!

And a student has contributed a customer care agent in community_contributions that uses ADK.